# Week 7 — Sequence-to-Sequence and Attention

The encoder–decoder paradigm and the attention mechanism that broke its information bottleneck. This is the conceptual bridge to the Transformer.

## Learning Objectives

- Implement an LSTM-based seq2seq model for a toy translation task.
- Identify the information-bottleneck failure of vanilla seq2seq empirically.
- Derive and implement Bahdanau (additive) and Luong (multiplicative) attention.
- Visualize attention heatmaps and interpret them linguistically.

## Required Reading

- Sutskever, I., Vinyals, O., & Le, Q. V. (2014). *Sequence to Sequence Learning with Neural Networks*.
- Bahdanau, D., Cho, K., & Bengio, Y. (2015). *Neural Machine Translation by Jointly Learning to Align and Translate*.
- Luong, M.-T., Pham, H., & Manning, C. D. (2015). *Effective Approaches to Attention-based Neural Machine Translation*.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. A toy translation task

We define a small synthetic task that captures the structure of translation without the data overhead. Numbers in English digit form → number reversed. For example: `one two three` → `three two one`.

This is hard for vanilla seq2seq when sequences get long, because the encoder must compress the entire input into one fixed-size vector.

In [ ]:
DIGITS = ['zero','one','two','three','four','five','six','seven','eight','nine']
PAD, BOS, EOS = '<pad>', '<s>', '</s>'
SPECIALS = [PAD, BOS, EOS]
VOCAB = SPECIALS + DIGITS
tok2id = {t: i for i, t in enumerate(VOCAB)}
id2tok = {i: t for t, i in tok2id.items()}
V = len(VOCAB)

def make_example(length):
    src = [np.random.randint(0, 10) for _ in range(length)]
    tgt = src[::-1]
    src_ids = [tok2id[DIGITS[d]] for d in src]
    tgt_ids = [tok2id[BOS]] + [tok2id[DIGITS[d]] for d in tgt] + [tok2id[EOS]]
    return src_ids, tgt_ids

def make_batch(N, length):
    pairs = [make_example(length) for _ in range(N)]
    src = torch.tensor([p[0] for p in pairs])      # (N, length)
    tgt = torch.tensor([p[1] for p in pairs])      # (N, length+2)
    return src, tgt

src, tgt = make_batch(2, 5)
print(f"src: {[id2tok[i.item()] for i in src[0]]}")
print(f"tgt: {[id2tok[i.item()] for i in tgt[0]]}")

## 2. Vanilla seq2seq

The encoder reads the source and produces a final hidden state $\mathbf{s}$. The decoder is initialized with $\mathbf{s}$ and generates the target autoregressively. Everything the decoder will ever know about the source must fit in this single vector — the **bottleneck**.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, V, emb=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=tok2id[PAD])
        self.rnn = nn.LSTM(emb, hidden, batch_first=True)

    def forward(self, src):
        e = self.emb(src)
        out, (h, c) = self.rnn(e)
        return out, (h, c)

class DecoderNoAttn(nn.Module):
    def __init__(self, V, emb=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=tok2id[PAD])
        self.rnn = nn.LSTM(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, V)

    def forward(self, tgt_in, state):
        e = self.emb(tgt_in)
        out, state = self.rnn(e, state)
        return self.out(out), state

class Seq2SeqNoAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder(V); self.dec = DecoderNoAttn(V)

    def forward(self, src, tgt_in):
        _, state = self.enc(src)
        logits, _ = self.dec(tgt_in, state)
        return logits

## 3. Bahdanau (additive) attention

At each decoder step $t$, compute alignment scores against every encoder state $\mathbf{h}_i$:

$$e_{t,i} = \mathbf{v}^\top \tanh(W_s \mathbf{s}_{t-1} + W_h \mathbf{h}_i),$$

$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}, \qquad \mathbf{c}_t = \sum_i \alpha_{t,i} \mathbf{h}_i.$$

The context vector $\mathbf{c}_t$ is concatenated with the decoder input to produce the next hidden state. The decoder is no longer limited to the single final encoder state — it can attend to any position.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.Ws = nn.Linear(hidden, hidden, bias=False)
        self.Wh = nn.Linear(hidden, hidden, bias=False)
        self.v  = nn.Linear(hidden, 1, bias=False)

    def forward(self, decoder_state, encoder_outputs, src_mask=None):
        # decoder_state: (B, H);  encoder_outputs: (B, S, H);  src_mask: (B, S) bool, True at PAD
        s = self.Ws(decoder_state).unsqueeze(1)           # (B, 1, H)
        h = self.Wh(encoder_outputs)                       # (B, S, H)
        e = self.v(torch.tanh(s + h)).squeeze(-1)         # (B, S)
        if src_mask is not None:
            e = e.masked_fill(src_mask, -1e9)
        alpha = F.softmax(e, dim=-1)                       # (B, S)
        ctx = torch.bmm(alpha.unsqueeze(1), encoder_outputs).squeeze(1)  # (B, H)
        return ctx, alpha

class DecoderAttn(nn.Module):
    def __init__(self, V, emb=32, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=tok2id[PAD])
        self.attn = BahdanauAttention(hidden)
        self.rnn = nn.LSTMCell(emb + hidden, hidden)
        self.out = nn.Linear(hidden * 2, V)
        self.hidden = hidden

    def forward(self, tgt_in, enc_out, state, src_mask=None):
        # Stepwise decoding to keep attention readable.
        h, c = state
        h, c = h.squeeze(0), c.squeeze(0)   # (B, H)
        B, T = tgt_in.shape
        logits_all, alphas_all = [], []
        for t in range(T):
            e = self.emb(tgt_in[:, t])
            ctx, alpha = self.attn(h, enc_out, src_mask)
            h, c = self.rnn(torch.cat([e, ctx], dim=-1), (h, c))
            logits = self.out(torch.cat([h, ctx], dim=-1))
            logits_all.append(logits); alphas_all.append(alpha)
        return torch.stack(logits_all, dim=1), torch.stack(alphas_all, dim=1)

class Seq2SeqAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder(V); self.dec = DecoderAttn(V)

    def forward(self, src, tgt_in):
        enc_out, (h, c) = self.enc(src)
        src_mask = (src == tok2id[PAD])
        logits, alphas = self.dec(tgt_in, enc_out, (h, c), src_mask)
        return logits, alphas

## 4. Training: attention vs. no-attention on long sequences

We train both models on the reversal task for sequences of length 4. Then we evaluate on lengths 4 and 10. The attention model should generalize far better.

In [ ]:
def train(model, lengths_train=(4,), n_steps=600, batch=64):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for step in range(n_steps):
        L = lengths_train[step % len(lengths_train)]
        src, tgt = make_batch(batch, L)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        out = model(src, tgt_in)
        logits = out[0] if isinstance(out, tuple) else out
        loss = F.cross_entropy(logits.reshape(-1, V), tgt_out.reshape(-1),
                               ignore_index=tok2id[PAD])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model, length, n=200):
    src, tgt = make_batch(n, length)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    with torch.no_grad():
        out = model(src, tgt_in)
        logits = out[0] if isinstance(out, tuple) else out
        preds = logits.argmax(-1)
    # Sequence-level accuracy: all non-PAD positions correct.
    mask = (tgt_out != tok2id[PAD])
    seq_correct = ((preds == tgt_out) | ~mask).all(dim=1).float().mean().item()
    return seq_correct

m_noattn = train(Seq2SeqNoAttn(), lengths_train=(4,))
m_attn   = train(Seq2SeqAttn(),   lengths_train=(4,))

print(f"{'length':>8} {'no-attn':>10} {'attn':>10}")
for L in [4, 6, 8, 10]:
    a1 = evaluate(m_noattn, L); a2 = evaluate(m_attn, L)
    print(f"{L:>8d} {a1:>10.3f} {a2:>10.3f}")
print("\nAttention generalizes to longer sequences far better — that's the bottleneck story.")

## 5. Visualizing attention

For a successful translation, attention should align the i-th output digit with the (L − i)-th input digit (since the task is reversal). The heatmap should show a clear anti-diagonal.

In [ ]:
src, tgt = make_batch(1, 6)
tgt_in = tgt[:, :-1]
with torch.no_grad():
    _, alphas = m_attn(src, tgt_in)
alpha = alphas[0].numpy()  # (T_tgt, T_src)

src_tokens = [id2tok[i.item()] for i in src[0]]
tgt_tokens = [id2tok[i.item()] for i in tgt_in[0]]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(alpha, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens)
ax.set_yticks(range(len(tgt_tokens))); ax.set_yticklabels(tgt_tokens)
ax.set_xlabel('source'); ax.set_ylabel('target (decoder step)')
ax.set_title('Bahdanau attention — should show anti-diagonal for reversal')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 6. Luong attention — multiplicative variants

Luong et al. (2015) proposed simpler scoring functions:

- **Dot:** $e_{t,i} = \mathbf{s}_t^\top \mathbf{h}_i$
- **General:** $e_{t,i} = \mathbf{s}_t^\top W \mathbf{h}_i$
- **Concat:** like Bahdanau

The dot variant requires no extra parameters and is faster — directly foreshadowing the **scaled dot-product attention** of the Transformer in Week 8.

In [ ]:
class LuongDotAttention(nn.Module):
    def forward(self, decoder_state, encoder_outputs, src_mask=None):
        # decoder_state: (B, H);  encoder_outputs: (B, S, H)
        e = torch.bmm(encoder_outputs, decoder_state.unsqueeze(-1)).squeeze(-1)  # (B, S)
        if src_mask is not None:
            e = e.masked_fill(src_mask, -1e9)
        alpha = F.softmax(e, dim=-1)
        ctx = torch.bmm(alpha.unsqueeze(1), encoder_outputs).squeeze(1)
        return ctx, alpha

# Sanity: same shape conventions as Bahdanau attention.
attn = LuongDotAttention()
B, S, H = 2, 6, 64
fake_state = torch.randn(B, H); fake_enc = torch.randn(B, S, H)
ctx, a = attn(fake_state, fake_enc)
print(f"context shape: {tuple(ctx.shape)}, attention shape: {tuple(a.shape)}, sum α = {a.sum(-1).tolist()}")

## 7. Exercises

1. **Length generalization.** Train seq2seq with and without attention on lengths $\{3, 4, 5\}$. Evaluate on lengths up to 20. Plot accuracy vs. length for both. Quantify how much attention buys you.
2. **Non-monotonic alignment.** Construct a small synthetic task with non-monotonic alignment (e.g., a permutation that is not a reversal). Visualize the attention heatmap and check whether it recovers the true permutation.
3. **Coverage attention.** Implement coverage attention (Tu et al., 2016): add a coverage vector $\mathbf{c}_t = \sum_{t' < t} \boldsymbol{\alpha}_{t'}$ as an input to the attention scorer. Show that it reduces over- and under-translation.
4. **Compare Bahdanau vs. Luong.** Train both on the same data, with matched parameter counts. Report final BLEU and training wall-clock time.

---

## Next Week

Week 8 — The Transformer, end-to-end. We replace recurrence with pure attention.